### 데이터 준비

#### 데이터 다운로드

In [ ]:
!pip install datasets # Package install -> 데이터 다운로드 명령어 구현

from datasets import load_dataset # hugging face 데이터셋 패키지 import
data = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset") # 데이터셋 다운로드

#### 데이터 구조 확인하기

In [ ]:
print("Data type: ", type(data)) # 데이터 타입 확인
print("Data structure: ", data) # 데이터 구조 확인
print("Data keys: ", data.keys()) # 데이터 키 확인

print(data['train'][0]) # 실제 데이터 확인

#### 데이터 정제하기

- 학습(train) / 검증(validation) / 테스트(test) 셋으로 데이터 구분

- 값이 없는 데이터 제거

In [ ]:
def remove_empty_data(row):
    return all(row[field] not in [None, ""] for field in ['id', 'text', 'label', 'sentiment']) # remove the null data

train_data = data['train'].filter(remove_empty_data) # null 데이터 제거
dev_data =data['dev'].filter(remove_empty_data) # null 데이터 제거
test_data = data['test'].filter(remove_empty_data) # null 데이터 제거

#### 데이터의 벡터 표현 생성

- 텍스트를 숫자로 바꾸기 (벡터화)
    - Bag of Words
        - 단어의 빈도수를 통해 문서를 표현하는 방법

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer # CountVectorizer : BoW 벡터화 구현
vectorizer = CountVectorizer() # 데이터를 벡터화 해주는 모델
vectorizer.fit(train_data['text']) # 텍스트 문서 모음을 토큰 수의 행렬로 변환

print(vectorizer.vocabulary_) # 텍스트 문서에 나타난 어휘의 집합을 출력
print(len(vectorizer.vocabulary_)) # 텍스트 문서에 나타난 어휘의 집합의 길이를 출력

train_vectors = vectorizer.transform(train_data['text']) # 학습 데이터를 숫자로 변환
dev_vectors = vectorizer.transform(dev_data['text']) # 검증 데이터를 숫자로 변환
test_vectors = vectorizer.transform(test_data['text']) # 테스트 데이터를 숫자로 변환

print(train_vectors[1234]) # 학습 데이터의 1234번째 벡터 출력
print(train_data['text'][1234]) # 학습 데이터의 1234번째 텍스트 출력

In [ ]:
sample_num = 17 # 확인하고 싶은 샘플 번호
sample_origin = train_data['text'][sample_num] # 샘플 번호에 해당하는 원본 텍스트
sample_transform = train_vectors[sample_num] # 확인하고 싶은 샘플의 vector로 변환된 결과
sample_inverse_transform = vectorizer.inverse_transform(sample_transform) # 벡터화된 결과를 다시 텍스트로 변환

print("original text: ", sample_origin) # 원본 텍스트 출력
print("Transformed Vector: ", sample_transform) # 벡터화된 결과 출력
print("Inverse Transformed Text: ", sample_inverse_transform) # 벡터화된 결과를 다시 텍스트로 변환한 결과 출력

### 모델 정의 및 컴파일

#### 다층 퍼셉트론 모델 정의

- optimizer : Adam
- Activation functn : GeLU
- Hidden Size: Layer1-100, Layer2-50
- The number of hidden layer : 2
- Learning rate : 1e-5
- Epoch number : 10
- Batch number : 128
- Loss function : Cross entropy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn

# randomness 제거
seed_num = 42

# fix pytorch random seed
torch.manual_seed(seed_num)
torch.cuda.manual_seed(seed_num)
torch.cuda.manual_seed_all(seed_num)

# fix cudnn random seed
cudnn.benchmark = False
cudnn.deterministic = True

###### Seed 란?

- seed는 난수 생성기를 초기화 하는데 사용하는 숫자이다.

- seed를 고정하는 이유는 재현성을 보장하기 위해서이다.

#### MLP 모델 정의

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__() # 모델의 레이어를 정의하고 초기화하는 부분
        # 첫번째 은닉층
        self.fc1 = nn.Linear(input_size, hidden_size)
        # 두번째 은닉층
        self.fc2 = nn.Linear(hidden_size, hidden_size//2) # 첫번째 hidden_size의 절반으로 줄임
        # 출력층
        self.fc3 = nn.Linear(hidden_size//2, output_size)
        # 활성화 함수
        self.activation = nn.GELU() # ReLU 보다 성능이 좋은 활성화 함수
        # 출력층의 활성화 함수
        self.output_activation = nn.Softmax(dim=1) 

    def forward(self, x): # 모델의 정방향 계산을 수행하는 메서드
        out1 = self.fc1(x)
        out2 = self.activation(out1)
        out3 = self.fc2(out2)
        out4 = self.activation(out3)
        out5 = self.fc3(out4)
        output = self.output_activation(out5)
        final_out = output
        return final_out

In [ ]:
# 하이퍼 파라미터 정의
input_size = len(vectorizer.vocabulary_) # input size
hidden_size = 1000 # hidden size
output_size = 3 # output size is 3 (positive / negative / neutral)
learning_rate = 0.001 # learning rate
batch_size = 128 
num_epochs = 10
loss_function = nn.CrossEntropyLoss() # 다중 클래스 분류 문제에서 자주 사용되는 손실 함수

# 모델 정의
model = MLP(input_size, hidden_size, output_size)
device = torch.device('cuda')
model = model.to(device)

# 하이퍼 파라미터 셋팅 - optimizer 정의
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# dev_vectors 와 dev_data['label']를 텐서로 변환 -> MLP 모델에 넣을 수 있는 형태로 변환
train_tensors = torch.cuda.FloatTensor(train_vectors.toarray(), device = device)
dev_tensors = torch.cuda.FloatTensor(dev_vectors.toarray(), device = device)
dev_labels = torch.tensor(dev_data['label'], dtype=torch.long, device=device)

# 학습 전 dev 성능 확인
dev_outputs = model(dev_tensors)
dev_preds = torch.argmax(dev_outputs, axis=1)
dev_accuracy = torch.sum(dev_preds == dev_labels).item() / len(dev_labels)
print(f"Epoch 0, Accuracy: {dev_accuracy:.4f}")

In [ ]:
best_accuracy = 0

for epoch in range(num_epochs):
    model.train() # model을 학습
    epoch_loss = 0 # epoch에서 발생하는 loss 초기화

    # batch size 단위로 학습 진행 --> batch size 만큼의 데이터에 대해 한번 파라미터 업데이트
    for i in range(0, len(train_tensors), batch_size):
        # batch 단위 데이터 생성
        batch_data = train_tensors[i:i+batch_size]
        batch_labels = torch.tensor(train_data['label'][i:i+batch_size], device=device)

        # 학습 시작
        # 1. 순전파
        outputs = model(batch_data)
        # 2. 오차 계산
        loss = loss_function(outputs, batch_labels)
        # 3. 역전파
        optimizer.zero_grad()
        loss.backward()
        # 4. 가중치 업데이트
        optimizer.step()

    # 현재 batch의 loss 값을 누적 (현재 epoch 내에서 epoch에 대한 평균 loss 계산용)
    epoch_loss += loss.item()
    # 매 epoch마다 dev 성능 측정
    model.eval() # 모델을 평가용으로 셋팅
    with torch.no_grad():
        dev_outputs = model(dev_tensors)
        dev_preds = torch.argmax(dev_outputs, axis=1)
        dev_accuracy = torch.sum(dev_preds == dev_labels).item() / len(dev_labels)

        # save best model on dev data
        if dev_accracy > best_accuracy:
            best_model = model
            best_accuracy = dev_accuracy

In [ ]:
# 테스트 세트로 모델 평가하기
from sklearn.metrics import accuacy_score # Accuracy 측정 함수 import

test_tensors = torch.cuda.FloatTensor(test_vectors.toarray(), device=device)
pred_results = best_model(test_tensors) # 최종 모델로 test 데이터 예측
pred_labels = torch.argmax(pred_results, axis=1)

accuracy = accuracy_score(test_data['label'], pred_labels.tolist()) # 정확도 측정
print("Accuracy: {:.2f}%".format(accuracy*100)) # 정확도 출력